# DNA Sequence Analysis - Query Examples

This Jupyter Notebook demonstrates how to run all DNA sequence analysis queries directly from Python.

**Contents:**
1. Sequence Comparison
2. Promoter Motif Search
3. Motif Quantity Analysis
4. Generate Test File
5. Inject Motif
6. Complete Analysis Workflow

## Setup: Import Required Modules

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from IPython.display import HTML, display

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import analysis modules
from modules import (
    compare_sequences,
    search_promoter_motif,
    search_motifs_quantity,
    generate_test_file,
    inject_motif
)

print("✓ All modules imported successfully")
print(f"✓ Working directory: {os.getcwd()}")

---
## 1. Sequence Comparison

Compare two DNA sequences with detailed analysis including:
- Triplet-by-triplet comparison with color coding
- Reverse complement analysis
- Motif search in reverse complement regions

In [ ]:
# Define two sequences to compare
sequence_1 = "ATGCTAGCTAGCTAGC"
sequence_2 = "ATGCAAGCTAGCTAGC"

print("Sequence 1 (Wild-type):")
print(sequence_1)
print("\nSequence 2 (Variant):")
print(sequence_2)
print("\nDifference at position 4-5: TAG → AAG")

In [ ]:
# Perform comparison
result_html = compare_sequences.display_results(
    sequence_1, 
    sequence_2,
    optimizer1="Wild-type (Original)",
    optimizer2="Variant (Mutated)"
)

# Display results
display(HTML(result_html))

In [ ]:
# Example 2: Another pair of sequences
seq_a = "ATTATAAATGCTAGC"
seq_b = "ATTGCAAATGCTAGC"

result_html_2 = compare_sequences.display_results(
    seq_a, 
    seq_b,
    optimizer1="Sequence A",
    optimizer2="Sequence B"
)

display(HTML(result_html_2))

---
## 2. Promoter Motif Search (ATTATA)

Search for the ATTATA motif in gene regions using:
- CSV file with gene locations (Start, Stop positions)
- FASTA file with genomic sequence

In [ ]:
# Check if example files exist
csv_path = "../examples/example_genes_full_genome.csv"
fasta_path = "../examples/example_E-coli.fasta"

files_exist = os.path.exists(csv_path) and os.path.exists(fasta_path)

if files_exist:
    print("✓ Example files found")
    
    # Load and preview data
    df_genes = pd.read_csv(csv_path, nrows=5)
    print("\nGene locations (first 5 rows):")
    print(df_genes)
else:
    print(f"⚠️  Example files not found at:")
    print(f"  - {csv_path}")
    print(f"  - {fasta_path}")
    print("\nPlease place example files in the 'examples/' directory")

In [ ]:
if files_exist:
    # Search for ATTATA motif
    print("Searching for ATTATA motif in promoter regions...\n")
    
    results = search_promoter_motif.process_files(csv_path, fasta_path)
    
    print(f"Results:")
    print(f"  Total ATTATA occurrences: {results['count']}")
    print(f"  Total positions searched: {results['total_searches']}")
    print(f"  Probability of occurrence: {results['probability']}")
    print(f"\n  First 20 indices where ATTATA was found:")
    if results['indices']:
        for i, idx in enumerate(results['indices'][:20]):
            print(f"    {i+1}. Position {idx}")
    else:
        print("    No ATTATA motifs found")

---
## 3. Motif Quantity Analysis

Analyze ATTATA motif occurrences in a protein/DNA database:
- Identify duplicate sequences
- Count motif occurrences
- Calculate probability statistics

In [ ]:
# Check for example database
db_path = "../examples/example_multiple_sequences.csv"

if os.path.exists(db_path):
    print("✓ Example database found")
    
    # Preview database
    df_db = pd.read_csv(db_path, nrows=5)
    print(f"\nDatabase preview (first 5 sequences):")
    print(df_db[['Serial', 'Id', 'organism_name']].to_string())
else:
    print(f"⚠️  Example database not found: {db_path}")

In [ ]:
if os.path.exists(db_path):
    # Analyze motif quantity
    print("Analyzing ATTATA motif in database...\n")
    
    results = search_motifs_quantity.search_motif(db_path)
    
    # Display summary
    print("Analysis Summary:")
    print(f"  Total sequences: {results['total_sequences']}")
    print(f"  Sequences with ATTATA: {results['num_sequences_with_motif']}")
    print(f"  Total ATTATA occurrences: {results['total_occurrences']}")
    print(f"  Total nucleotides: {results['total_nucleotides']}")
    print(f"\n  Probability Metrics:")
    print(f"    Per nucleotide: {results['prob_nucleotide']}")
    print(f"    Per sequence: {results['prob_sequence']}")
    print(f"\n  Database Status:")
    print(f"    {results['uniqueness_message']}")

In [ ]:
if os.path.exists(db_path):
    # Display results table
    print("\nSequences containing ATTATA motif:")
    display(HTML(results['results_df']))

In [ ]:
if os.path.exists(db_path):
    # Check generated output files
    print("\nGenerated Output Files:")
    
    if os.path.exists("../cleaned_database.csv"):
        df_clean = pd.read_csv("../cleaned_database.csv")
        print(f"✓ cleaned_database.csv - {len(df_clean)} sequences (duplicates removed)")
    
    if os.path.exists("../duplicates_report.csv"):
        df_dupes = pd.read_csv("../duplicates_report.csv")
        if len(df_dupes) > 0:
            print(f"✓ duplicates_report.csv - {len(df_dupes)} duplicate sequences found")
            print("\nFirst few duplicates:")
            display(df_dupes.head())
        else:
            print(f"✓ duplicates_report.csv - No duplicates found")

---
## 4. Generate Test File

Create synthetic DNA sequences with potential ATTATA motif injection sites

In [ ]:
# Generate test sequences
print("Generating 30 synthetic sequences...\n")

test_file = "generated_test_sequences.csv"
generate_test_file.generate_test_file(total_sequences=30, output_path=test_file)

# Load and display results
df_test = pd.read_csv(test_file)

print(f"✓ Generated {len(df_test)} sequences")
print(f"✓ Saved to: {test_file}")
print(f"\nColumns: {', '.join(df_test.columns.tolist())}")
print(f"\nDataset statistics:")
print(f"  DNA sequences: {len(df_test)} total")
print(f"  Average DNA length: {df_test['predicted_dna'].str.len().mean():.0f} bp")
print(f"  Average protein length: {df_test['protein_sequence'].str.len().mean():.1f} aa")

In [ ]:
# Display first 5 sequences
print("First 5 generated sequences:")
print(df_test[['predicted_dna', 'protein_sequence', 'expected_case']].head().to_string())

In [ ]:
# Analyze expected cases
print("\nExpected ATTATA injection cases distribution:")
for idx, row in df_test.head(10).iterrows():
    print(f"\nSequence {idx+1}:")
    print(f"  Expected: {row['expected_case']}")
    print(f"  DNA: {row['predicted_dna'][:30]}...")
    print(f"  Protein: {row['protein_sequence']}")

---
## 5. Inject Motif (ATTATA)

Inject ATTATA motif into DNA sequences while preserving protein sequences using synonymous codons

In [ ]:
# Inject motif into test sequences
print("Injecting ATTATA motif into test sequences...\n")

input_csv = "generated_test_sequences.csv"
output_csv = "sequences_with_injected_motif.csv"

print("Injection strategies:")
print("  Case 1: ATT + ATA → ATTATA (Isoleucine + Isoleucine)")
print("  Case 2: ?A + TTA + TA? → ATTATA (Alanine + Leucine + Tyrosine)")
print("  Case 3: ?AT + TAT + A? → ATTATA (Asparagine + Tyrosine + Methionine)")
print()

results_inject = inject_motif.process_csv(input_csv, output_csv)

print(f"✓ Processing complete")
print(f"✓ Output saved to: {output_csv}")

In [ ]:
# Load and display injected sequences
df_injected = pd.read_csv(output_csv)

print(f"Injected sequences summary:")
print(f"  Total sequences: {len(df_injected)}")
print(f"  Columns: {', '.join(df_injected.columns.tolist())}")
print(f"\nFirst 5 injected sequences:")
print(df_injected.head().to_string())

In [ ]:
# Compare original vs injected sequences
print("Comparison of original vs injected sequences:\n")

for idx in range(min(3, len(df_injected))):
    orig_row = df_test.iloc[idx]
    inj_row = df_injected.iloc[idx]
    
    print(f"Sequence {idx + 1}:")
    print(f"  Original DNA:  {orig_row['predicted_dna'][:40]}...")
    print(f"  Injected DNA:  {inj_row['predicted_dna'][:40]}...")
    print(f"  Protein (orig): {orig_row['protein_sequence']}")
    print(f"  Protein (inj):  {inj_row['protein_sequence']}")
    
    # Verify protein is unchanged
    proteins_match = orig_row['protein_sequence'] == inj_row['protein_sequence']
    print(f"  Protein preserved: {'✓' if proteins_match else '✗'}")
    print()

In [ ]:
# Verify ATTATA is present in injected sequences
print("Verification: ATTATA presence in injected sequences\n")

attata_count = 0
sequences_with_attata = 0

for idx, row in df_injected.iterrows():
    count = row['predicted_dna'].count('ATTATA')
    if count > 0:
        attata_count += count
        sequences_with_attata += 1

print(f"Results:")
print(f"  Total ATTATA found: {attata_count}")
print(f"  Sequences with ATTATA: {sequences_with_attata}/{len(df_injected)}")
print(f"  Injection success rate: {(sequences_with_attata/len(df_injected))*100:.1f}%")

---
## 6. Complete Analysis Workflow

End-to-end pipeline: Generate → Inject → Analyze

In [ ]:
print("="*70)
print("COMPLETE ANALYSIS PIPELINE")
print("="*70)
print()

# Step 1: Generate test sequences
print("STEP 1: Generate Test Sequences")
print("-" * 70)
pipeline_test_file = "pipeline_test_sequences.csv"
print(f"Creating 50 synthetic sequences...")
generate_test_file.generate_test_file(50, pipeline_test_file)
df_pipeline_test = pd.read_csv(pipeline_test_file)
print(f"✓ Generated {len(df_pipeline_test)} sequences\n")

In [ ]:
# Step 2: Inject ATTATA motif
print("STEP 2: Inject ATTATA Motif")
print("-" * 70)
pipeline_injected_file = "pipeline_injected_sequences.csv"
print(f"Injecting motif into sequences...")
inject_motif.process_csv(pipeline_test_file, pipeline_injected_file)
df_pipeline_injected = pd.read_csv(pipeline_injected_file)
print(f"✓ Modified {len(df_pipeline_injected)} sequences\n")

In [ ]:
# Step 3: Analyze motif distribution
print("STEP 3: Analyze Motif Distribution")
print("-" * 70)
print(f"Searching for ATTATA in modified sequences...")
pipeline_results = search_motifs_quantity.search_motif(pipeline_injected_file)
print(f"✓ Analysis complete\n")

# Display results
print("ANALYSIS RESULTS:")
print("-" * 70)
print(f"Total sequences analyzed: {pipeline_results['total_sequences']}")
print(f"Sequences with ATTATA: {pipeline_results['num_sequences_with_motif']}")
print(f"Total ATTATA occurrences: {pipeline_results['total_occurrences']}")
print(f"Total nucleotides: {pipeline_results['total_nucleotides']}")
print(f"\nProbability Metrics:")
print(f"  Per nucleotide: {pipeline_results['prob_nucleotide']}")
print(f"  Per sequence: {pipeline_results['prob_sequence']}")
print(f"\nDatabase Quality:")
print(f"  {pipeline_results['uniqueness_message']}")

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("WORKFLOW SUMMARY")
print("="*70)

total_changes = 0
attata_found = 0

# Count ATTATA occurrences
for idx, row in df_pipeline_injected.iterrows():
    attata_found += row['predicted_dna'].count('ATTATA')

print(f"\nFiles Generated:")
print(f"  1. {pipeline_test_file} - {len(df_pipeline_test)} test sequences")
print(f"  2. {pipeline_injected_file} - {len(df_pipeline_injected)} injected sequences")
print(f"  3. cleaned_database.csv - {len(pd.read_csv('../cleaned_database.csv'))} unique sequences")

print(f"\nMotif Statistics:")
print(f"  ATTATA occurrences: {attata_found}")
print(f"  Successful injections: {int(pipeline_results['num_sequences_with_motif'])} sequences")
print(f"  Success rate: {(int(pipeline_results['num_sequences_with_motif'].replace(',', ''))/len(df_pipeline_injected))*100:.1f}%")

print(f"\n✓ Workflow completed successfully!")

---
## Summary

This notebook demonstrated all major DNA sequence analysis capabilities:

1. **Sequence Comparison** - Compare and analyze differences between sequences
2. **Promoter Motif Search** - Find ATTATA in gene regions from genomic data
3. **Motif Quantity Analysis** - Analyze motif distribution in sequence databases
4. **Test File Generation** - Create synthetic sequences for validation
5. **Motif Injection** - Insert ATTATA while preserving protein sequences
6. **Complete Pipeline** - End-to-end analysis workflow

All functions preserve biological accuracy by:
- Using the standard genetic code for translation
- Employing synonymous codons to maintain protein sequences
- Implementing proper DNA/RNA complement calculations

For more details, see the [README.md](README.md) in the queries directory.